# FOWT Simulation Agent

This notebook shows the agentic framework without the Streamlit UI. The architecture is deliberately small:

**natural language → LLM tool call → registered engineering tool → deterministic FMU simulation**

`agent.py` owns orchestration and run state. `simulation.py` owns the physics and FMU execution. The LLM never accesses the FMU directly.

## 1. Install dependencies

Run this once in a fresh environment.

In [10]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configure an OpenAI-compatible LLM

The default is a local Ollama server at `http://localhost:11434/v1` using `qwen2.5`. If that is what you use, no environment variables are required.

For another OpenAI-compatible backend, set the three variables below before creating the client. Do not commit API keys to the notebook.

In [2]:
# Optional example for a remote OpenAI-compatible backend:
# import os
# os.environ["LLM_BASE_URL"] = "https://your-provider.example/v1"
# os.environ["LLM_API_KEY"] = "..."
# os.environ["LLM_MODEL"] = "your-model"

## 3. Import the framework

The notebook and Streamlit app use the same `agent.py`.

In [ ]:
from agent import (
    TOOL_SCHEMAS,
    RunStore,
    backend_label,
    compare_runs,
    get_client,
    get_model,
    plot_results,
    run_agent,
    run_simulation,
)

client = get_client()
model = get_model()
store = RunStore()

print(backend_label())

Ollama · qwen2.5


## 4. Inspect the tool boundary

These schemas are the complete set of actions exposed to the LLM. Adding a new capability means registering another deterministic tool, not giving the model unrestricted access to Python or the FMU.

In [4]:
[(tool["function"]["name"], tool["function"]["description"]) for tool in TOOL_SCHEMAS]

[('run_simulation', 'Run and store the FOWT FMU for one wind/wave condition.'),
 ('plot_results',
  'Plot stored signals: surge, sway, heave, roll, pitch, yaw, tension1, tension2, tension3.'),
 ('compare_runs', 'Compare one summary metric across stored runs.')]

## 5. The deterministic path, without an LLM

Calling the engineering tool directly makes the boundary explicit. The same function is what the agent invokes through tool calling.

In [5]:
direct = run_simulation(
    store,
    Hs=2.5,
    Tp=10.0,
    U_hub=12.0,
    TI=0.10,
    duration=10.0,
)
direct

{'run_id': 'run1',
 'reused': False,
 'surge_rms_m': 1.7750417721554246,
 'pitch_rms_deg': 2.052730586487836,
 'max_tension_mn': 1.232655027539062,
 'constraint_ok': True}

The raw time series stay in `RunStore`; only the compact parameters and summary are intended for LLM context.

In [6]:
store.runs[direct["run_id"]]["params"], store.runs[direct["run_id"]]["summary"]

({'Hs': 2.5, 'Tp': 10.0, 'U_hub': 12.0, 'TI': 0.1, 'duration': 10.0},
 {'surge_rms_m': 1.7750417721554246,
  'pitch_rms_deg': 2.052730586487836,
  'max_tension_mn': 1.232655027539062,
  'constraint_ok': True})

## 6. Let the agent orchestrate the simulation

Now the LLM is responsible for interpreting the sentence and deciding which registered tools to call. `trace` contains only the observable tool calls, not private model reasoning.

### What `run_agent()` actually does

Before running it, here is the whole loop, trimmed to the essentials. Not runnable on its own — just the mechanism:

```python
while True:
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=TOOL_SCHEMAS,
    )
    message = response.choices[0].message

    if not message.tool_calls:
        return message.content

    messages.append(message)
    for call in message.tool_calls:
        args   = json.loads(call.function.arguments)
        result = dispatch(call.function.name, args, store)
        messages.append({
            "role":         "tool",
            "tool_call_id": call.id,
            "content":      json.dumps(result),
        })
```

That loop **is** the agent. The rest of `agent.py` is tools, validation, plotting, and state. `run_agent()` in the next cell is the same thing with a `max_steps` safety cap and a `trace` list for observability.

In [7]:
text, figures, trace = run_agent(
    "Run Hs 6 m, Tp 10 s, 12 m/s wind, TI 0.15 for 10 s and plot pitch",
    store,
    client=client,
    model=model,
)

print(text)
print("\nTool calls:")
for call in trace:
    print("  ", call)

for fig in figures:
    fig.show()

The simulation for Hs 6 m, Tp 10 s, 12 m/s wind, TI 0.15 for 10 seconds has been run successfully and the pitch angle RMS is plotted below.

Here are the summary metrics from the new run:
- Surge RMS: 2.147 m
- Pitch RMS: 2.088°
- Maximum Tension: 1.251 MN
- Constraints were satisfied (constraint_ok: true)

![](pitch_plot_run2.png)

Tool calls:
   run_simulation({"Hs": 6, "Tp": 10, "U_hub": 12, "TI": 0.15, "duration": 10})
   plot_results({"run_id": "run2", "signals": ["pitch"]})


## 7. Follow up using stored simulation state

The same `RunStore` is reused, so the agent can operate on earlier results instead of starting from a blank conversation every time.

In [8]:
text, figures, trace = run_agent(
    "Compare maximum mooring tension between run1 and run2",
    store,
    client=client,
    model=model,
)

print(text)
print("\nTool calls:")
for call in trace:
    print("  ", call)

ongyang
{"name": "compare_runs", "arguments": {"run_ids": ["run1", "run2"], "metric": "max_tension_mn"}}

Tool calls:


## 8. Use the tools manually when useful

Agentic orchestration is optional. Engineers can always call the deterministic tools directly.

In [9]:
comparison = compare_runs(store, ["run1", "run2"], "max_tension_mn")
plot_results(store, "run2", ["surge", "tension1"])
manual_figures = store.drain_figures()

print(comparison)
for fig in manual_figures:
    fig.show()

{'metric': 'max_tension_mn', 'values': {'run1': 1.232655027539062, 'run2': 1.251354000576864}}


## What this notebook demonstrates

The LLM owns interpretation and tool selection. Python owns validation, execution and persistent simulation state. `simulation.py` owns the numerical mechanics and platform-specific FMU. The framework can therefore change LLM backends without changing the engineering simulation, and engineering capabilities can grow by adding registered tools.